# LoanGuard AI - Covenant Breach Predictor Training

This notebook trains an XGBoost model to predict covenant breaches with SHAP explanations.

**Run on Google Colab**: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/loanguard-ai/blob/main/notebooks/train_breach_predictor.ipynb)

In [ ]:
# Install dependencies
!pip install -q xgboost shap pandas numpy scikit-learn google-cloud-bigquery google-cloud-storage

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pickle
import json

## 1. Load Training Data

Option A: From BigQuery (production)
Option B: From CSV (development)

In [ ]:
# Option A: Load from BigQuery (uncomment to use)
# from google.colab import auth
# auth.authenticate_user()
# 
# from google.cloud import bigquery
# client = bigquery.Client(project='your-project-id')
# 
# query = """
# SELECT 
#     m.loan_id,
#     m.debt_to_ebitda_ratio,
#     m.interest_coverage_ratio,
#     m.current_ratio,
#     m.net_worth,
#     m.measurement_date,
#     l.industry_sector,
#     l.loan_type,
#     l.borrower_rating,
#     c.threshold_value,
#     CASE WHEN m.debt_to_ebitda_ratio > c.threshold_value THEN 1 ELSE 0 END as breach
# FROM `loanguard_data.covenant_measurements` m
# JOIN `loanguard_data.loans` l ON m.loan_id = l.loan_id
# JOIN `loanguard_data.covenants` c ON m.loan_id = c.loan_id
# WHERE c.covenant_type = 'DEBT_TO_EBITDA'
# """
# df = client.query(query).to_dataframe()

In [ ]:
# Option B: Generate synthetic training data
np.random.seed(42)
n_samples = 10000

# Generate features
data = {
    'debt_to_ebitda_ratio': np.random.lognormal(1.0, 0.5, n_samples),
    'interest_coverage_ratio': np.random.lognormal(1.5, 0.6, n_samples),
    'current_ratio': np.random.lognormal(0.5, 0.3, n_samples),
    'net_worth': np.random.lognormal(18, 1.5, n_samples),
    'revenue_growth': np.random.normal(0.05, 0.15, n_samples),
    'ebitda_margin': np.random.normal(0.15, 0.08, n_samples),
    'debt_service_coverage': np.random.lognormal(0.5, 0.4, n_samples),
    'quick_ratio': np.random.lognormal(0.3, 0.3, n_samples),
    'operating_cash_flow': np.random.lognormal(16, 1.5, n_samples),
    'capex_to_revenue': np.random.uniform(0.02, 0.15, n_samples),
    'days_since_last_measurement': np.random.randint(1, 90, n_samples),
    'measurement_volatility': np.random.uniform(0.05, 0.4, n_samples),
    'industry_risk_score': np.random.uniform(1, 10, n_samples),
    'industry_sector': np.random.choice(['Technology', 'Healthcare', 'Manufacturing', 'Retail', 'Energy'], n_samples),
    'loan_type': np.random.choice(['Term Loan', 'Revolver', 'Bridge Loan'], n_samples),
    'covenant_type': np.random.choice(['DEBT_TO_EBITDA', 'INTEREST_COVERAGE', 'CURRENT_RATIO'], n_samples),
    'borrower_rating': np.random.choice(['AAA', 'AA', 'A', 'BBB', 'BB', 'B'], n_samples),
    'geographic_region': np.random.choice(['North America', 'Europe', 'Asia', 'LATAM'], n_samples),
}

df = pd.DataFrame(data)

# Generate target: breach probability based on features
# Higher D/E, lower ICR, lower current ratio -> higher breach probability
breach_score = (
    0.3 * (df['debt_to_ebitda_ratio'] > 4.0).astype(float) +
    0.2 * (df['debt_to_ebitda_ratio'] > 5.0).astype(float) +
    0.2 * (df['interest_coverage_ratio'] < 2.0).astype(float) +
    0.15 * (df['current_ratio'] < 1.0).astype(float) +
    0.1 * (df['revenue_growth'] < -0.1).astype(float) +
    0.05 * (df['industry_risk_score'] > 7).astype(float) +
    np.random.uniform(0, 0.1, n_samples)  # noise
)

df['breach'] = (breach_score > 0.4).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Breach rate: {df['breach'].mean():.2%}")

In [ ]:
# Display sample data
df.head()

## 2. Feature Engineering

In [ ]:
# Define feature columns
numeric_features = [
    'debt_to_ebitda_ratio',
    'interest_coverage_ratio',
    'current_ratio',
    'net_worth',
    'revenue_growth',
    'ebitda_margin',
    'debt_service_coverage',
    'quick_ratio',
    'operating_cash_flow',
    'capex_to_revenue',
    'days_since_last_measurement',
    'measurement_volatility',
    'industry_risk_score',
]

categorical_features = [
    'industry_sector',
    'loan_type',
    'covenant_type',
    'borrower_rating',
    'geographic_region',
]

target = 'breach'

In [ ]:
# Convert categorical features to category dtype for XGBoost
for col in categorical_features:
    df[col] = df[col].astype('category')

# Save category mappings for inference
category_mappings = {}
for col in categorical_features:
    category_mappings[col] = list(df[col].cat.categories)

print("Category mappings:")
for col, cats in category_mappings.items():
    print(f"  {col}: {cats}")

In [ ]:
# Prepare features and target
feature_cols = numeric_features + categorical_features
X = df[feature_cols]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 3. Train XGBoost Model

In [ ]:
# Create DMatrix with categorical support
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)

# XGBoost parameters
params = {
    'objective': 'binary:logistic',
    'eval_metric': ['logloss', 'auc'],
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'seed': 42,
}

# Train with early stopping
evals = [(dtrain, 'train'), (dtest, 'test')]
model = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50,
)

## 4. Evaluate Model

In [ ]:
# Make predictions
y_pred_proba = model.predict(dtest)
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate metrics
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
}

print("\nModel Performance:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Feature importance
importance = model.get_score(importance_type='gain')
importance_df = pd.DataFrame([
    {'feature': k, 'importance': v}
    for k, v in importance.items()
]).sort_values('importance', ascending=False)

print("\nTop 10 Feature Importance (Gain):")
print(importance_df.head(10).to_string(index=False))

## 5. Generate SHAP Explainer

In [ ]:
# Create SHAP explainer
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for test set (sample for efficiency)
sample_size = min(1000, len(X_test))
X_sample = X_test.sample(sample_size, random_state=42)
shap_values = explainer.shap_values(xgb.DMatrix(X_sample, enable_categorical=True))

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base probability): {explainer.expected_value:.4f}")

In [ ]:
# SHAP summary plot
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=True)

In [ ]:
# SHAP bar plot
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, plot_type='bar', show=True)

## 6. Save Model and Explainer

In [ ]:
# Save model
model.save_model('breach_predictor.json')
print("Model saved to breach_predictor.json")

# Save as pickle (alternative)
with open('breach_predictor.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved to breach_predictor.pkl")

# Save SHAP explainer
with open('breach_predictor_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)
print("SHAP explainer saved to breach_predictor_explainer.pkl")

# Save category mappings
with open('train_categories.json', 'w') as f:
    json.dump(category_mappings, f, indent=2)
print("Category mappings saved to train_categories.json")

# Save model metadata
metadata = {
    'model_type': 'xgboost',
    'model_version': '1.0.0',
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'target': target,
    'metrics': metrics,
    'expected_value': float(explainer.expected_value),
    'best_iteration': model.best_iteration,
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("Metadata saved to model_metadata.json")

## 7. Upload to Google Cloud Storage

In [ ]:
# Upload to GCS (uncomment to use)
# from google.cloud import storage
# 
# bucket_name = 'loanguard-models'
# client = storage.Client()
# bucket = client.bucket(bucket_name)
# 
# files_to_upload = [
#     ('breach_predictor.pkl', 'models/breach_predictor.pkl'),
#     ('breach_predictor_explainer.pkl', 'shap_explainers/breach_predictor_explainer.pkl'),
#     ('train_categories.json', 'categories/train_categories.json'),
#     ('model_metadata.json', 'metadata/breach_predictor_metadata.json'),
# ]
# 
# for local_path, gcs_path in files_to_upload:
#     blob = bucket.blob(gcs_path)
#     blob.upload_from_filename(local_path)
#     print(f"Uploaded {local_path} to gs://{bucket_name}/{gcs_path}")
# 
# print("\nAll files uploaded to GCS!")

## 8. Test Inference

In [ ]:
# Test inference with a sample
sample_input = {
    'debt_to_ebitda_ratio': 4.5,
    'interest_coverage_ratio': 1.8,
    'current_ratio': 0.9,
    'net_worth': 50000000,
    'revenue_growth': -0.05,
    'ebitda_margin': 0.12,
    'debt_service_coverage': 1.1,
    'quick_ratio': 0.7,
    'operating_cash_flow': 5000000,
    'capex_to_revenue': 0.08,
    'days_since_last_measurement': 45,
    'measurement_volatility': 0.25,
    'industry_risk_score': 6.5,
    'industry_sector': 'Manufacturing',
    'loan_type': 'Term Loan',
    'covenant_type': 'DEBT_TO_EBITDA',
    'borrower_rating': 'BB',
    'geographic_region': 'North America',
}

# Create DataFrame
sample_df = pd.DataFrame([sample_input])
for col in categorical_features:
    sample_df[col] = sample_df[col].astype('category')

# Predict
sample_dmatrix = xgb.DMatrix(sample_df[feature_cols], enable_categorical=True)
probability = model.predict(sample_dmatrix)[0]

print(f"\nSample Prediction:")
print(f"  Breach Probability: {probability:.2%}")
print(f"  Risk Level: {'HIGH' if probability > 0.5 else 'MEDIUM' if probability > 0.25 else 'LOW'}")

# SHAP explanation
shap_values_sample = explainer.shap_values(sample_dmatrix)
print(f"\nTop Risk Factors:")
shap_df = pd.DataFrame({
    'feature': feature_cols,
    'shap_value': shap_values_sample[0],
}).sort_values('shap_value', ascending=False, key=abs)
print(shap_df.head(5).to_string(index=False))